# 🧪 Cuaderno 01: Exploración Inicial y Validación del Dataset
### Predicción de Entalpías de Combustión Mediante Machine Learning
**Autora:** Karol Paola Camacho López  
**Facultad de Ingeniería Química — Benemérita Universidad Autónoma de Puebla (BUAP)**  
**Fecha:** Septiembre 2026

---

## 🎯 Objetivos de este Cuaderno
1. Verificar el entorno de ejecución (`.venv`) y las versiones de las librerías científicas.
2. Cargar el dataset crudo desde `data/raw/` usando rutas relativas robustas.
3. Realizar una primera auditoría de dimensiones, nombres de columnas, tipos de datos y valores faltantes (`NaN`).
4. Probar las funciones estequiométricas moleculares de ChemPy.

In [ ]:
# ==============================================================================
# 1. COMPROBACIÓN DEL ENTORNO DE LIBRERÍAS
# ==============================================================================
import os
import sys
import numpy as np
import pandas as pd
import chempy
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Entorno de ejecución validado exitosamente.")
print(f"-> Python:       {sys.version.split()[0]}")
print(f"-> NumPy:        {np.__version__}")
print(f"-> Pandas:       {pd.__version__}")
print(f"-> ChemPy:       {chempy.__version__}")
print(f"-> Scikit-Learn: {sklearn.__version__}")

## 2. Carga del Dataset con Ruta Relativa
Siguiendo la regla de oro del manual, utilizamos rutas relativas basadas en el directorio de trabajo para garantizar portabilidad total entre computadoras.

In [ ]:
# Definimos la ruta relativa hacia el archivo de datos crudos
ruta_datos = os.path.join("..", "data", "raw", "base_entalpias.csv")

if os.path.exists(ruta_datos):
    print(f"-> Archivo localizado en: {ruta_datos}")
    df = pd.read_csv(ruta_datos)
    print(f"-> Dimensiones: {df.shape[0]} filas (moléculas) y {df.shape[1]} columnas (variables).")
else:
    print(f"⚠️ AVISO: Aún no has colocado el archivo en '{ruta_datos}'.")
    print("Creando un DataFrame ilustrativo para verificar las funciones del manual...")
    # DataFrame demostrativo con valores termodinámicos experimentales reales
    datos_demo = {
        'nombre': ['Metano', 'Etano', 'Propano', 'Butano', 'Etanol', 'Acido Acetico', 'Benceno', 'Tolueno'],
        'formula': ['CH4', 'C2H6', 'C3H8', 'C4H10', 'C2H6O', 'C2H4O2', 'C6H6', 'C7H8'],
        'SMILES': ['C', 'CC', 'CCC', 'CCCC', 'CCO', 'CC(=O)O', 'c1ccccc1', 'Cc1ccccc1'],
        'dH_combustion': [-890.8, -1560.7, -2220.0, -2877.6, -1366.8, -874.2, -3267.6, -3910.2]
    }
    df = pd.DataFrame(datos_demo)
    print(f"-> Dataset demo cargado con {len(df)} compuestos de prueba.")

# Visualizar las primeras 5 filas
df.head()

## 3. Diagnóstico Estructural y Detección de Datos Nulos
Inspeccionamos los tipos de datos de cada columna y comprobamos si existen celdas vacías (`NaN`).

In [ ]:
print("--- INFORMACIÓN ESTRUCTURAL (TIPOS DE DATOS) ---")
df.info()

print("\n--- AUDITORÍA DE VALORES NULOS ---")
print(df.isnull().sum())

print("\n--- RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS ---")
df.describe()

## 4. Feature Engineering Químico con ChemPy
Probamos el módulo `src.funciones_quimicas` para enriquecer la tabla con masas molares, balance de combustión y número de átomos.

In [ ]:
# Importar las funciones creadas en src/
import sys
sys.path.append(os.path.abspath(os.path.join("..")))
from src.funciones_quimicas import extraer_descriptores_estequiometricos, balancear_combustion

# Ejemplo individual de balanceo para el Etanol
reac, prod = balancear_combustion('C2H6O')
print("Combustión balanceada del Etanol:", reac, "-->", prod)

# Aplicar masivamente la extracción a cada fila del DataFrame
features_quimicas = df['formula'].apply(extraer_descriptores_estequiometricos)
df_enriquecido = pd.concat([df, features_quimicas], axis=1)
df_enriquecido.head()

## 5. Visualización Inicial de la Variable Objetivo ($\Delta H_c^\circ$)

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 4))

# Gráfica de dispersión: Masa Molar vs Entalpía de Combustión
plt.subplot(1, 2, 1)
sns.scatterplot(data=df_enriquecido, x='masa_molar', y='dH_combustion', color='crimson', s=70)
plt.title("Masa Molar vs $\Delta H_c^\circ$", fontweight="bold")
plt.xlabel("Masa Molar (g/mol)")
plt.ylabel("$\Delta H_c^\circ$ (kJ/mol)")

# Histograma de distribución de entalpías
plt.subplot(1, 2, 2)
sns.histplot(df_enriquecido['dH_combustion'], kde=True, color='teal', bins=6)
plt.title("Distribución de $\Delta H_c^\circ$", fontweight="bold")
plt.xlabel("$\Delta H_c^\circ$ (kJ/mol)")

plt.tight_layout()
plt.show()